# Hands-on: create and run the Job from code

This is the automation version of everything you'd otherwise click through in Workflows > Jobs > Create Job. Great live moment for the team: show the UI walkthrough once, then show this notebook creating the exact same Job in one cell.

**Before running:** import `01_ingest_data.ipynb`, `02_transform_data.ipynb`, and `03_validate_and_notify.ipynb` into your workspace first, then set `NOTEBOOK_FOLDER` below to the folder you put them in.

In [ ]:
# databricks-sdk ships preinstalled on most current runtimes; this is a safe no-op if so
%pip install -q databricks-sdk --upgrade

In [ ]:
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

# Inside a Databricks notebook, WorkspaceClient() authenticates automatically
# using the notebook's own execution context - no token to paste in.
w = WorkspaceClient()
print(f"Connected as: {w.current_user.me().user_name}")

In [ ]:
# Set this to the workspace folder where you imported the three task notebooks
NOTEBOOK_FOLDER = "/Workspace/Users/<your-email>/workflows_demo"

# Optional: notify address for the demo (change to a real address/Slack webhook)
NOTIFY_EMAIL = "<your-email>@yourcompany.com"

print(f"Tasks will be loaded from: {NOTEBOOK_FOLDER}")

## Create the multi-task Job
One call defines: three tasks with dependencies, job-level parameters, per-task retries, an 8am daily schedule, and email notifications.

In [ ]:
created_job = w.jobs.create(
    name="workflows-demo-sales-pipeline",
    tags={"purpose": "team-training-demo"},

    # Job-level parameters -> flow into any task's dbutils.widgets of the same name
    parameters=[
        jobs.JobParameterDefinition(name="min_trip_distance", default="1.0"),
        jobs.JobParameterDefinition(name="force_fail", default="false"),
    ],
    tasks=[
        jobs.Task(
            task_key="ingest_data",
            notebook_task=jobs.NotebookTask(notebook_path=f"{NOTEBOOK_FOLDER}/01_ingest_data"),
            max_retries=2,
            min_retry_interval_millis=5 * 60 * 1000,  # 5 minute gap between retries
        ),
        jobs.Task(
            task_key="transform_data",
            depends_on=[jobs.TaskDependency(task_key="ingest_data")],
            notebook_task=jobs.NotebookTask(notebook_path=f"{NOTEBOOK_FOLDER}/02_transform_data"),
        ),
        jobs.Task(
            task_key="validate",
            depends_on=[jobs.TaskDependency(task_key="transform_data")],
            notebook_task=jobs.NotebookTask(notebook_path=f"{NOTEBOOK_FOLDER}/03_validate_and_notify"),
            max_retries=2,
            min_retry_interval_millis=60 * 1000,  # 1 minute gap - good for a live demo
        ),
    ],
    # 8am daily trigger
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 8 * * ?",
        timezone_id="Asia/Kolkata",
        pause_status=jobs.PauseStatus.PAUSED,  # paused so it doesn't fire mid-demo; unpause when ready
    ),
    email_notifications=jobs.JobEmailNotifications(
        on_success=[NOTIFY_EMAIL],
        on_failure=[NOTIFY_EMAIL],
    ),
)

print(f"Created job_id={created_job.job_id}")
print(f"Open it: {w.config.host}/jobs/{created_job.job_id}")

## Trigger a run right now

`run_now` lets you override job parameters per-run - here's how you'd force the `validate` task to fail on purpose to show retries live, without editing the Job definition.

In [ ]:
run = w.jobs.run_now(
    job_id=created_job.job_id,
    job_parameters={"min_trip_distance": "1.0", "force_fail": "true"},  # demo mode: forces a retry
)
print(f"Started run_id={run.run_id}")
print(f"Watch it live: {w.config.host}/jobs/{created_job.job_id}/runs/{run.run_id}")

In [ ]:
import time

run_id = run.run_id
while True:
    status = w.jobs.get_run(run_id=run_id)
    state = status.state.life_cycle_state
    print(f"State: {state}")
    if state in (jobs.RunLifeCycleState.TERMINATED, jobs.RunLifeCycleState.SKIPPED,
                 jobs.RunLifeCycleState.INTERNAL_ERROR):
        print(f"Result: {status.state.result_state}")
        break
    time.sleep(15)

## Re-run clean (force_fail = false)

Same job, same code, just a different parameter value - shows the whole DAG going green end to end.

In [ ]:
clean_run = w.jobs.run_now(
    job_id=created_job.job_id,
    job_parameters={"min_trip_distance": "1.0", "force_fail": "false"},
)
print(f"Started clean run_id={clean_run.run_id}")
print(f"Watch it live: {w.config.host}/jobs/{created_job.job_id}/runs/{clean_run.run_id}")

## Enable the schedule for real (optional)

The Job was created paused so it wouldn't fire mid-session. Run this only if you actually want the 8am daily run to go live after the training.

In [ ]:
# Uncomment to activate the schedule
# w.jobs.update(
#     job_id=created_job.job_id,
#     new_settings=jobs.JobSettings(
#         schedule=jobs.CronSchedule(
#             quartz_cron_expression="0 0 8 * * ?",
#             timezone_id="Asia/Kolkata",
#             pause_status=jobs.PauseStatus.UNPAUSED,
#         )
#     ),
# )
print("Uncomment above to turn the daily schedule on.")

## Cleanup

Run this after the demo so you don't leave a stray Job (and its paused schedule) sitting in the workspace.

In [ ]:
# Uncomment to delete the demo job
# w.jobs.delete(job_id=created_job.job_id)
print("Uncomment above to delete the demo job when the session is over.")

### Talking points for the team
- Everything clicked in the Workflows UI has a 1:1 SDK/REST API equivalent - useful for CI/CD, where Jobs are defined in code and deployed via Databricks Asset Bundles rather than hand-clicked per environment.
- `run_now(job_parameters=...)` overriding the job's default parameters per-run is exactly how you'd trigger backfills or region-specific runs without editing the Job definition each time.
- The Job was created **paused** on purpose - always do this for demo/test jobs so a forgotten schedule doesn't keep firing after the session ends.